# ASSIST & IAS15 interpolation

The underlying numerical integrator for ASSIST is IAS15 (Rein & Liu 2015), a 15th order predictor-corrector integrator with an adaptive step-size.
The primary method provided by ASSIST to evolve simulations, `assist.Extras.integrate_or_interpolate`, which uses the fact that for each timestep, ASSIST determines a polynomial that provides state data within the step **without the need to manually integrate a smaller step size**.

Below, we dissect this behaviour to explain how it can be taken advantage of for significant performance improvements, e.g. in the `Sorcha` package's ephemeris generation module (see https://arxiv.org/html/2506.02140v1#S2.T1)

### A simple example to demonstrate interpolation

First, let's import REBOUND, ASSIST as well as numpy and matplotlib, and load up the ephemeris files.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import rebound
import assist 

ephem = assist.Ephem("../data/linux_p1550p2650.440", "../data/sb441-n16.bsp")

We create a simple rebound simulation and attach ASSIST. We'll add Asteroid Holman, at JD 2458849.5 (arbitrary) from JPL Horizons.

In [15]:
T0 = 2458849.5
T0_sim = T0 - ephem.jd_ref

def setup_sim():
    sim = rebound.Simulation()
    ax = assist.Extras(sim, ephem)
    holman_initial = rebound.Particle(
        x=3.338875348598862E+00, y=-9.176518412197102E-01, z=-5.038590741719294E-01, 
        vx=2.805663364339457E-03, vy=7.550408665778840E-03, vz=2.980028207875623E-03)
    sim.add(holman_initial)
    sim.t = T0_sim
    return sim, ax

sim, ax = setup_sim()

print(f"Initialized simulation at t={sim.t}, initial position: {sim.particles[0].xyz}")

Initialized simulation at t=7304.5, initial position: [3.338875348598862, -0.9176518412197102, -0.5038590741719294]


In the standard pattern for ASSIST, you drive the simulation by selecting a set of times and calling `ax.integrate_or_interpolate` on those times. Let's look at what's happening under the hood.

To start, we'll ask for assist to step forward 1 hour (~0.05 days).

In [16]:
T1 = T0_sim + 0.05
ax.integrate_or_interpolate(T1)
sim_time_at_t1 = sim.t

print(f"Called `integrate_or_interpolate({T1})`, sim.t={sim.t}, position: {sim.particles[0].xyz}")

Called `integrate_or_interpolate(7304.55)`, sim.t=7304.605583077408, position: [3.3390156030441953, -0.917274312858135, -0.503710068412949]


Great -- ASSIST has given us the position at T1, but note that the internal simulation time is higher than our requested step. 
Let's say we want to find our state at `T2 = T1 + 0.05` (another hour).

In [17]:
T2 = T1 + 0.05
ax.integrate_or_interpolate(T2)
sim_time_at_t2 = sim.t
print(f"Called `integrate_or_interpolate({T2})`, sim.t={sim.t}, position: {sim.particles[0].xyz}")
print(f"Difference between internal simulation time (sim.t@T1 - sim.t@T2)={sim_time_at_t1 - sim_time_at_t2}")

Called `integrate_or_interpolate(7304.6)`, sim.t=7304.605583077408, position: [3.3391558000425747, -0.9168967686444428, -0.5035610539585716]
Difference between internal simulation time (sim.t@T1 - sim.t@T2)=0.0


ASSIST evolved our state, but our simulation's internal time is still the same as when we called `integrate_or_interpolate` with T1.

This is a demonstration of the **interpolate** part of the `integrate_or_interpolate` function: once IAS15 has converged a particular timestep and we want to compute state data at a time within that step, we don't need to integrate again - we can use an internal polynomial representation stored by IAS15 to calculate that state data much faster (exactly how much faster depends on a number of circumstances, but the reader is referred to the [`Sorcha` paper](https://arxiv.org/html/2506.02140v1#S2.T1) where some explanantion of benchmarking technique is done).

To better understand the mechanics of this, we will manually step forward a simulation (using the adaptive timestep cadence of IAS15), and demonstrate how to interpolate within a timestep.

Let us begin by resetting our simulation and collecting the values we'll need to interpolate:


In [25]:
sim, ax = setup_sim()

initial_state = [*sim.particles[0].xyz, *sim.particles[0].vxyz]
print(f"Initialized simulation at t={sim.t}, initial position: {sim.particles[0].xyz}")

# We call integrate_or_interpolate on the same time as the simulation, to populate
# ax.current_state and ax.last_state.
ax.integrate_or_interpolate(sim.t)
print("\nPopulated ax last_state and current_state...")
print(f"Stepped simulation forward; sim.t={sim.t}, sim.particles xyz:{sim.particles[0].xyz}, vxyz:{sim.particles[0].vxyz}, axyz:{[sim.particles[0].ax, sim.particles[0].ay, sim.particles[0].az]}\n")
print(f"ax.last_state:    xyz: {ax.last_state[0].xyz}, vxyz:{ax.last_state[0].vxyz}, axyz: {[ax.last_state[0].ax, ax.last_state[0].ay, ax.last_state[0].az]}\n")
print(f"ax.current_state: xyz: {ax.current_state[0].xyz}, vxyz:{ax.current_state[0].vxyz}, axyz: {[ax.current_state[0].ax, ax.current_state[0].ay, ax.current_state[0].az]}\n")

print("\nNow calling sim.step...")
sim.step()

print(f"Stepped simulation forward; sim.t={sim.t}, sim.particles xyz:{sim.particles[0].xyz}, vxyz:{sim.particles[0].vxyz}, axyz:{[sim.particles[0].ax, sim.particles[0].ay, sim.particles[0].az]}\n")
print(f"ax.last_state:    xyz: {ax.last_state[0].xyz}, vxyz:{ax.last_state[0].vxyz}, axyz: {[ax.last_state[0].ax, ax.last_state[0].ay, ax.last_state[0].az]}\n")
print(f"ax.current_state: xyz: {ax.current_state[0].xyz}, vxyz:{ax.current_state[0].vxyz}, axyz: {[ax.current_state[0].ax, ax.current_state[0].ay, ax.current_state[0].az]}\n")

# Let's extract the state data we'd need to interpolate within this step, i.e. between sim.t=7304.5 and sim.t=7304.501
b = sim.ri_ias15._br
state_data_for_interpolation = np.array([
    *ax.last_state[0].xyz,
    *ax.last_state[0].vxyz,
    ax.last_state[0].ax,
    ax.last_state[0].ay,
    ax.last_state[0].az,
    b.p0[0],b.p0[1],b.p0[2], # b1 xyz
    b.p1[0],b.p1[1],b.p1[2], # b2 xyz
    b.p2[0],b.p2[1],b.p2[2], # b3 xyz
    b.p3[0],b.p3[1],b.p3[2], # b4 xyz
    b.p4[0],b.p4[1],b.p4[2], # b5 xyz
    b.p5[0],b.p5[1],b.p5[2], # b6 xyz
    b.p6[0],b.p6[1],b.p6[2], # b7 xyz
])

Initialized simulation at t=7304.5, initial position: [3.338875348598862, -0.9176518412197102, -0.5038590741719294]

Populated ax last_state and current_state...
Stepped simulation forward; sim.t=7304.5, sim.particles xyz:[3.338875348598862, -0.9176518412197102, -0.5038590741719294], vxyz:[0.002805663364339457, 0.00755040866577884, 0.002980028207875623], axyz:[0.0, 0.0, 0.0]

ax.last_state:    xyz: [3.338875348598862, -0.9176518412197102, -0.5038590741719294], vxyz:[0.002805663364339457, 0.00755040866577884, 0.002980028207875623], axyz: [0.0, 0.0, 0.0]

ax.current_state: xyz: [3.338875348598862, -0.9176518412197102, -0.5038590741719294], vxyz:[0.002805663364339457, 0.00755040866577884, 0.002980028207875623], axyz: [0.0, 0.0, 0.0]


Now calling sim.step...
Stepped simulation forward; sim.t=7304.501, sim.particles xyz:[3.3388781542507373, -0.9176442908078727, -0.503856094141982], vxyz:[0.0028056403862606154, 0.007550415009266075, 0.0029800316870872267], axyz:[-2.2978085617450198e-05, 6.3

### Interpolating within a timestep
Alright, we've extracted all the data to interpolate within a step. The `br` values above are from IAS15, which has a converged set of coefficients, called `b` coefficients (see [IAS15 paper](https://arxiv.org/abs/1409.4779)). We extract the acceleration values from `ax.last_state`.

#### A note about acceleration
Currently, the correct acceleration for interpolating within a timestep is the acceleration is in `ax.last_state` (as seen in the output above). That is to say, the state in `sim.particles` is now updated to be the value at the end of the timestep, and we need the values from the initial state. In practice, we can use the values from `sim.particles.{xyz,vxyz}` from before we call `sim.step()`, but *we cannot use the acceleration from there*. The only correct acceleration is the value in `ax.last_state` after a step is complete.

This same pattern appears in `assist_interpolate_simulation` (src/assist.c:512), where we collect state from two different 'blob's (steps) - the initial data from the beginning of a step, and the acceleration from the next step (which is the acceleration at the *beginning* of the next step, which is what is stored in `ax.last_state`). We explore this in the "SimulationArchives" notebook.

Calling `sim.step()` *does* update `ax.last_state` but does not update `ax.current_state`. So the correct state we need to interpolate within a timestep consists of:

1. the state data from `ax.last_state`, which contains the state vector at the *beginning* of our timestep (position, velocity) and acceleration as materialized by the simulation step procedure,
2. the internal **end** time and step size of the simulation, `sim.t` and `sim.dt_last_done`, and
3. the b-coefficients (from `sim.ri_ias15._br`, or equivalently `ax._sim.ri_ias15._br`).

We will now demonstrate that this collection of data is indeed valid to interpolate within a timestep.

#### Working out an example

We'll use the same 'in-between' time as above: T_BTW = (sim.t - T0_sim)/2

In [11]:
T_BTW = sim.t - (sim.t - T0_sim)/2
print(T_BTW)
print(f"Is T_BTW within our step?: {T0_sim < T_BTW < sim.t}")

# Reference values
sim_ref, ax_ref = setup_sim()
ax_ref.integrate_or_interpolate(T_BTW)
ref = np.array([*sim_ref.particles[0].xyz, *sim_ref.particles[0].vxyz])

print(f"Calculated reference state at T_BTW={T_BTW}: {ref}")

7304.5005
Is T_BTW within our step?: True
Calculated reference state at T_BTW=7304.5005: [ 3.33887675e+00 -9.17648066e-01 -5.03857584e-01  2.80565188e-03
  7.55041184e-03  2.98002995e-03]


In [12]:
# Set up interpolation function manually
# Reference: ../src/assist.c

def evaluate_with_b(t, simt, dt_last_done, instate):
        h = np.float64(1-((simt - t) / dt_last_done))
        _s = np.zeros((1,9))
        _sv = np.zeros((1,8))

        assert (simt - dt_last_done) < t < simt

        s=_s[0]
        s[0] = dt_last_done * h
        s[1] = s[0] * s[0] / 2.
        s[2] = s[1] * h / 3.
        s[3] = s[2] * h / 2.
        s[4] = 3. * s[3] * h / 5.
        s[5] = 2. * s[4] * h / 3.
        s[6] = 5. * s[5] * h / 7.
        s[7] = 3. * s[6] * h / 4.
        s[8] = 7. * s[7] * h / 9.

        sv=_sv[0]
        sv[0] = dt_last_done * h;
        sv[1] =      sv[0] * h / 2.;
        sv[2] = 2. * sv[1] * h / 3.;
        sv[3] = 3. * sv[2] * h / 4.;
        sv[4] = 4. * sv[3] * h / 5.;
        sv[5] = 5. * sv[4] * h / 6.;
        sv[6] = 6. * sv[5] * h / 7.;
        sv[7] = 7. * sv[6] * h / 8.;

        outputs = np.zeros((1,6))

        # -- for each particle
        output=outputs[0]
        x,y,z,vx,vy,vz,ax,ay,az,b0x,b0y,b0z,b1x,b1y,b1z,b2x,b2y,b2z,b3x,b3y,b3z,b4x,b4y,b4z,b5x,b5y,b5z,b6x,b6y,b6z = instate

        output[0] = x + (s[8]*b6x + s[7]*b5x + s[6]*b4x + s[5]*b3x + s[4]*b2x + s[3]*b1x + s[2]*b0x + s[1]*ax + s[0]*vx );
        output[1] = y + (s[8]*b6y + s[7]*b5y + s[6]*b4y + s[5]*b3y + s[4]*b2y + s[3]*b1y + s[2]*b0y + s[1]*ay + s[0]*vy );
        output[2] = z + (s[8]*b6z + s[7]*b5z + s[6]*b4z + s[5]*b3z + s[4]*b2z + s[3]*b1z + s[2]*b0z + s[1]*az + s[0]*vz );

        output[3] = vx + sv[7]*b6x + sv[6]*b5x + sv[5]*b4x + sv[4]*b3x + sv[3]*b2x + sv[2]*b1x + sv[1]*b0x + sv[0]*ax;
        output[4] = vy + sv[7]*b6y + sv[6]*b5y + sv[5]*b4y + sv[4]*b3y + sv[3]*b2y + sv[2]*b1y + sv[1]*b0y + sv[0]*ay;
        output[5] = vz + sv[7]*b6z + sv[6]*b5z + sv[5]*b4z + sv[4]*b3z + sv[3]*b2z + sv[2]*b1z + sv[1]*b0z + sv[0]*az;
   
        return output

In [10]:
interpolated_state = evaluate_with_b(T_BTW, sim.t, sim.dt_last_done, state_data_for_interpolation)
print(f"Calculated interpolated state at T_BTW={T_BTW}: {interpolated_state}")

print(f"Maximum difference between the states: {np.max(interpolated_state - ref)}")

Calculated interpolated state at T_BTW=7304.5005: [ 3.33887675e+00 -9.17648066e-01 -5.03857584e-01  2.80565188e-03
  7.55041184e-03  2.98002995e-03]
Maximum difference between the states: 0.0


As you can see, the value from `integrate_or_interpolate` is exactly this procedure. This value is always a little bit off from just `sim.integrate`, but the latter can cause IAS15 to take uncharacteristically small timesteps which can adversely affect accuracy [citation needed]. 